# FELIX chr22 pilot — M1 calibration gate

Compares the four chr22 pilot arms on the same cohort and variant set:
FELIX limited / full (`felix/wdl/FelixPilot.wdl`) vs the existing Tractor-Mix
limited / full runs. Pass: λGC ≈ 1 on null phenotypes for all four models and
`comparability_flags.tsv` status `OK` (`felix/eval/README.md`).

**Variant set:** both FELIX configs scan the same FLARE-marker VCF the
Tractor-Mix pilot used (`felix/configs/felix.pilot.inputs.*.json.example`), so
differences come from the test, not the variants. M1b repeats FELIX on the full
long-read callset (`phase_vcf` + `flare_vcf`).

Set the four workflow roots (the `gs://…/submissions/<id>/<Workflow>/<wf-id>/`
directory of each finished run) in the config cell or via env vars.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (
    Path.cwd() / "aou-lr-phase-2" / "scripts",
    Path.cwd().parent / "aou-lr-phase-2" / "scripts",
    Path.home() / "AoU_DRC_LongReads_PhaseTwo_Storage" / "edit" / "aou-lr-phase-2" / "scripts",
    Path.cwd() / "scripts",
    Path.cwd().parent / "scripts",
    Path.cwd().parent.parent / "scripts",
):
    if (_d / "terra_notebook.py").is_file() and (_d.parent / "felix" / "scripts" / "compare_felix_tractor_calibration.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py")
from workspace_paths import data_root

FELIX_SCRIPTS = SCRIPTS.parent / "felix" / "scripts"
if not (FELIX_SCRIPTS / "compare_felix_tractor_calibration.py").is_file():
    raise FileNotFoundError(f"{FELIX_SCRIPTS} missing; run 00_sync_repo (needs the git clone)")
ROOT = data_root()
OUT = ROOT / "felix_pilot_gate"
OUT.mkdir(parents=True, exist_ok=True)
print("OUT:", OUT)
print("felix scripts:", FELIX_SCRIPTS)


## Config


In [ ]:
import os

RUN_ROOTS = {
    "felix_limited": os.environ.get("FELIX_LIMITED_ROOT", ""),
    "felix_full": os.environ.get("FELIX_FULL_ROOT", ""),
    "tractor_limited": os.environ.get("TRACTOR_LIMITED_ROOT", ""),
    "tractor_full": os.environ.get("TRACTOR_FULL_ROOT", ""),
}
SUFFIX = {"felix": ".felix.tsv", "tractor": ".tractor_mix.tsv"}
missing = [k for k, v in RUN_ROOTS.items() if not v.startswith("gs://")]
if missing:
    raise ValueError(f"set workflow roots for: {missing}")
for k, v in RUN_ROOTS.items():
    print(f"{k:16s} {v}")


## Fetch per-phenotype results


In [ ]:
import subprocess
from pathlib import Path

local_dirs = {}
for arm, root in RUN_ROOTS.items():
    suffix = SUFFIX[arm.split("_")[0]]
    listing = subprocess.run(
        ["gsutil", "ls", "-r", root.rstrip("/") + "/**"],
        check=True, capture_output=True, text=True,
    ).stdout.split()
    uris = sorted(u for u in listing if "/results_by_phenotype/" in u and u.endswith(suffix))
    if not uris:
        raise FileNotFoundError(f"{arm}: no */results_by_phenotype/*{suffix} under {root}")
    dest = OUT / "results" / arm
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "-m", "-q", "cp", *uris, str(dest) + "/"])
    local_dirs[arm] = dest
    print(f"{arm}: {len(uris)} phenotype TSVs -> {dest}")


## Run the M1 comparison


In [ ]:
import sys
import pandas as pd

GATE_OUT = OUT / "chr22_felix_tractor_gate"
subprocess.check_call([
    sys.executable, str(FELIX_SCRIPTS / "compare_felix_tractor_calibration.py"),
    "--felix-limited-dir", str(local_dirs["felix_limited"]),
    "--felix-full-dir", str(local_dirs["felix_full"]),
    "--tractor-limited-dir", str(local_dirs["tractor_limited"]),
    "--tractor-full-dir", str(local_dirs["tractor_full"]),
    "--out-dir", str(GATE_OUT),
])
for name in ("comparability_flags.tsv", "lambda_gc_wide.tsv"):
    path = GATE_OUT / name
    if path.is_file():
        print(name)
        display(pd.read_csv(path, sep="\t"))


## Reading the gate

- `comparability_flags.tsv` must say `OK` before λGC differences mean anything
  (matched effective N and tested-variant counts).
- λGC well above 1 for FELIX but not Tractor-Mix on the same phenotype points at
  the FELIX null / covariates; above 1 for all four points at the cohort or
  phenotype. Record the outcome in `felix/PLAN.md` (M1 row).
